# Hands-On Lab: Building and Training an HMM for Metamorphic Malware Detection

**Dataset:** IDAN1.csv, IDAN2.csv, IDAN3.csv (disassembled x86 opcode sequences)  

---

### Overview

Metamorphic malware rewrites its own code on each infection cycle, changing its binary signature while preserving its behaviour. Traditional signature-based detection fails against it. Hidden Markov Models (HMMs) attack the problem differently — instead of matching byte patterns, they model the **statistical transitions between instruction types**, which are harder to mutate away.

This lab trains a `MultinomialHMM` on x86 opcode sequences extracted from disassembled binaries and builds a classifier that scores new sequences by log-likelihood.

### Learning Objectives
1. Preprocess opcode sequences correctly for HMM input
2. Understand why sequence structure matters — an HMM learns *transitions*, not isolated opcodes
3. Train and validate a Hidden Markov Model with `hmmlearn`
4. Classify new opcode sequences by log-likelihood scoring
5. Critically evaluate the limitations of an unsupervised threshold-based approach

## Step 1: Install and Import Required Libraries

In [ ]:
!pip install hmmlearn --quiet

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder
from hmmlearn import hmm
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load and Inspect the Data

In [ ]:
data1 = pd.read_csv('IDAN1.csv')
data2 = pd.read_csv('IDAN2.csv')
data3 = pd.read_csv('IDAN3.csv')

print("Columns in data1:", data1.columns.tolist())
print("Columns in data2:", data2.columns.tolist())
print("Columns in data3:", data3.columns.tolist())

print(f"\nRows — IDAN1: {len(data1)}, IDAN2: {len(data2)}, IDAN3: {len(data3)}")
print("\nSample from IDAN1:")
print(data1.head())

## Step 3: Preprocess Opcode Sequences

**Critical design decision:** each CSV file represents one complete disassembled program — one sequence.

A common mistake is to split each opcode string (e.g. `"mov"`) into a list, producing thousands of length-1 sequences. This destroys all transition information. An HMM learns *how states transition* — it needs to see the full opcode flow through a program, not isolated individual instructions.

Correct approach: treat each file as a single sequence of opcodes.

In [ ]:
opcode_sequence1 = data1['Opcode'].astype(str).tolist()
opcode_sequence2 = data2['Opcode'].astype(str).tolist()
opcode_sequence3 = data3['Opcode'].astype(str).tolist()

# Three sequences total — one per binary
combined_opcode_sequences = [opcode_sequence1, opcode_sequence2, opcode_sequence3]

print(f"Number of sequences: {len(combined_opcode_sequences)}")
print(f"Sequence lengths: {[len(s) for s in combined_opcode_sequences]}")
print(f"\nFirst 10 opcodes of IDAN1: {opcode_sequence1[:10]}")

In [ ]:
label_encoder = LabelEncoder()
all_opcodes = [opcode for sequence in combined_opcode_sequences for opcode in sequence]
label_encoder.fit(all_opcodes)

print(f"Vocabulary size (unique opcodes): {len(label_encoder.classes_)}")
print(f"Sample opcodes: {list(label_encoder.classes_[:15])}")

encoded_opcode_sequences = [
    label_encoder.transform(sequence) for sequence in combined_opcode_sequences
]

print(f"\nEncoded Opcode Sequences Sample (first 20 of IDAN1):")
print(encoded_opcode_sequences[0][:20])

## Step 4: Concatenate Sequences and Define Lengths

`hmmlearn` expects a single concatenated array of observations plus a list of lengths that tells it where each sequence begins and ends.

In [ ]:
concatenated_sequences = np.concatenate(encoded_opcode_sequences)

sequence_lengths = [len(seq) for seq in encoded_opcode_sequences]

print(f"Total observations: {len(concatenated_sequences)}")
print(f"Sequence lengths: {sequence_lengths}")
print(f"Concatenated sequences (first 20): {concatenated_sequences[:20]}")

## Step 5: Build the HMM Model

Two hidden states represent two latent behavioural modes — conceptually one for "malware-like" patterns and one for "legitimate-like" patterns. The model will learn the optimal state assignments from the data.

In [ ]:
n_components = 2

# MultinomialHMM is appropriate here — observations are discrete opcode integers
model = hmm.MultinomialHMM(n_components=n_components, n_iter=100, random_state=42)

# Set initial start probabilities — equal chance of starting in either state
model.startprob_ = np.array([0.5, 0.5])

# Transition matrix — states tend to persist (0.7) but can switch (0.3)
model.transmat_ = np.array([
    [0.7, 0.3],
    [0.3, 0.7]
])

print("HMM model initialised.")
print(f"Hidden states: {n_components}")
print(f"Start probabilities: {model.startprob_}")
print(f"Initial transition matrix:\n{model.transmat_}")

## Step 6: Train the HMM Model

In [ ]:
# Train — reshape to (n_observations, 1) as required by hmmlearn
model.fit(concatenated_sequences.reshape(-1, 1), sequence_lengths)

print("Training complete.")
print(f"\nLearned transition matrix:\n{model.transmat_}")

## Step 7: Validate and Fix the Transition Matrix

With only 3 short sequences, some state transitions may never be observed during training, leaving zero-sum rows in `transmat_`. These would cause numerical errors at inference time and must be smoothed.

In [ ]:
def reinitialize_transmat(transmat, epsilon=1e-5):
    """Replace zero-sum rows with uniform distribution to ensure valid probabilities."""
    for i in range(transmat.shape[0]):
        if transmat[i].sum() == 0:
            print(f"  Row {i} has zero sum — reinitialising to uniform distribution")
            transmat[i] = np.full(transmat.shape[1], 1.0 / transmat.shape[1])
    return transmat

model.transmat_ = reinitialize_transmat(model.transmat_)

# Fix startprob_ if NaN
if np.isnan(model.startprob_).any():
    print("startprob_ contains NaN — reinitialising to uniform distribution")
    model.startprob_ = np.full(n_components, 1.0 / n_components)

# Verify startprob_ sums to 1
if not np.isclose(model.startprob_.sum(), 1.0):
    raise ValueError(f"startprob_ must sum to 1 (got {model.startprob_.sum()})")

print(f"startprob_ sum: {model.startprob_.sum():.4f} — OK")

## Step 8: Validate the Full Model

In [ ]:
def check_transmat(model):
    """Validate that all HMM parameters satisfy hmmlearn's internal constraints."""
    try:
        model._check()
        print("Model parameters are valid.")
    except ValueError as e:
        print(f"Validation error: {e}")

check_transmat(model)

# Score the training sequences to establish a baseline
print("\nLog-likelihood scores for training sequences:")
for i, (seq, length) in enumerate(zip(encoded_opcode_sequences, sequence_lengths)):
    score = model.score(seq.reshape(-1, 1))
    print(f"  IDAN{i+1} ({length} opcodes): {score:.2f}")

## Step 9: Classify New Opcode Sequences

New sequences are scored by log-likelihood — how probable are they under the learned model? Sequences that score well resemble the training data; sequences that score poorly are anomalous.

In [ ]:
def classify_opcode_sequence(opcode_sequence, trained_model, label_encoder, threshold=-50):
    """
    Classify an opcode sequence as Malware or Legit based on log-likelihood.
    
    Parameters
    ----------
    opcode_sequence : list of str
        Raw opcode strings e.g. ['mov', 'push', 'add']
    trained_model   : fitted MultinomialHMM
    label_encoder   : fitted LabelEncoder (same one used at training time)
    threshold       : float
        Log-likelihood below this value is classified as Malware.
        Should be calibrated empirically from known labelled samples.
    """
    try:
        encoded   = label_encoder.transform(opcode_sequence)
        reshaped  = encoded.reshape(-1, 1)
        log_score = trained_model.score(reshaped)
        label     = "Malware" if log_score < threshold else "Legit"
        return label, log_score
    except Exception as e:
        return f"Error: {str(e)}", None


# Example — short opcode sequence
new_sequence = ["mov", "add", "jmp", "push"]
label, score = classify_opcode_sequence(new_sequence, model, label_encoder)
print(f"Sequence: {new_sequence}")
print(f"Log-likelihood: {score:.2f}")
print(f"Classification: {label}")

# Test with a longer realistic sequence from IDAN1
test_sequence = opcode_sequence1[10:50]  # 40 opcodes from a real binary
label2, score2 = classify_opcode_sequence(test_sequence, model, label_encoder)
print(f"\nReal sequence from IDAN1 (40 opcodes):")
print(f"Log-likelihood: {score2:.2f}")
print(f"Classification: {label2}")

---

## Personal Analysis

### The Sequence Structure Problem

The most important implementation decision in this lab is treating each CSV file as a **single sequence** rather than a collection of individual opcodes. An HMM is fundamentally a model of sequential data — it learns the probability of transitioning from one hidden state to another. If you feed it thousands of length-1 sequences (one opcode each), you destroy all transition information and the model learns nothing meaningful. The Baum-Welch algorithm that trains the HMM needs to observe opcodes flowing through a program to learn behavioural patterns.

This connects directly to why HMMs are used for metamorphic malware detection in the first place: metamorphic engines change the specific instructions but tend to preserve the overall *flow* of operations — setup, payload, cleanup. The transition structure captures that flow.

### The Unsupervised Threshold Problem

This lab trains a single HMM on all three files combined with no ground-truth labels indicating which are malware and which are legitimate. The `-50` log-likelihood threshold in `classify_opcode_sequence()` is therefore arbitrary — there is no principled basis for it from this training setup.

A production-grade approach would require:
1. **Two separate models** — one trained on known malware sequences, one on known benign sequences
2. **Likelihood ratio scoring** — classify based on `score(malware_model) - score(benign_model)`
3. **Threshold calibration** on a labelled held-out set to optimise the precision/recall trade-off

The Christodorescu & Jha (2004) paper that established HMMs for metamorphic malware detection used exactly this two-model approach.

### Why HMMs Are Appropriate for This Problem

Metamorphic malware uses techniques like dead code insertion, register reassignment, and instruction substitution to evade signature detection. These transformations change the surface appearance but preserve the semantic sequence of operations. HMMs model this well because:
- The hidden states can represent semantic behavioural modes (e.g. initialisation, encryption, network communication) regardless of which specific instructions implement them
- The emission probabilities capture which opcodes are likely in each mode
- The transition matrix captures the order in which modes occur — which is harder to mutate than individual instructions